# P0-5 · Toán cho AI: đại số tuyến tính, giải tích, xác suất

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

In [ ]:
import numpy as np
e = {"vua":        np.array([0.9, 0.8, 0.1]),
     "hoàng hậu":  np.array([0.85, 0.75, 0.2]),
     "quả táo":    np.array([0.1, 0.2, 0.9])}
cos = lambda a, b: a @ b / (np.linalg.norm(a) * np.linalg.norm(b))
cos(e["vua"], e["hoàng hậu"])   # 0.996  rất giống
cos(e["vua"], e["quả táo"])     # 0.303  khác xa

0.9957846018984898
0.3034263294974635


In [ ]:
A = np.array([[1, 2], [3, 4]]); B = np.array([[5, 6], [7, 8]])
A @ B        # [[19 22] [43 50]]
B @ A        # [[23 34] [31 46]]   ← khác! nhân ma trận không giao hoán
A * B        # [[ 5 12] [21 32]]   ← * là nhân từng phần tử, không phải nhân ma trận

array([[19, 22],
       [43, 50]])
array([[23, 34],
       [31, 46]])
array([[ 5, 12],
       [21, 32]])


In [ ]:
X = np.random.rand(128, 784)   # batch 128 ảnh, mỗi ảnh 784 số
W = np.random.rand(256, 784)   # nn.Linear(784, 256) lưu weight shape (out, in)
b = np.random.rand(256)
H = X @ W.T + b                # (128, 784) @ (784, 256) → (128, 256), cộng b nhờ broadcasting

In [ ]:
f = lambda x: x ** 2
(f(3 + 1e-5) - f(3 - 1e-5)) / 2e-5      # 6.00000000004  khớp đạo hàm thật 6

6.000000000039306


## Bài tập 3: Kiểm tra gradient bằng số {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np

def numeric_grad(f, w, h=1e-5):
    g = np.zeros_like(w)
    for i in range(len(w)):
        e = np.zeros_like(w); e[i] = h
        g[i] = (f(w + e) - f(w - e)) / (2 * h)
    return g

sigmoid = lambda z: 1 / (1 + np.exp(-z))
x, y = np.array([0.5, -1.2, 2.0]), 1.0
loss = lambda w: -(y * np.log(sigmoid(w @ x)) + (1 - y) * np.log(1 - sigmoid(w @ x)))
w = np.array([0.1, 0.3, -0.2])
analytic = (sigmoid(w @ x) - y) * x
print(np.abs(numeric_grad(loss, w) - analytic).max())   # cỡ 1e-10

1.2864820320146464e-11


## Bài tập 4: Gợi ý phim bằng cosine similarity {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
phim = ["Avengers", "Titanic", "Toy Story", "Inception", "Frozen"]
nguoi = ["Lan", "Minh", "Hoa", "Tuấn"]
R = np.array([[5, 0, 4, 5, 0],     # Lan
              [4, 1, 5, 4, 5],     # Minh
              [1, 5, 0, 2, 4],     # Hoa
              [0, 4, 1, 0, 5]],    # Tuấn
             dtype=float)
# TODO: cos (độ giống của Lan với từng người), giong_nhat, goi_y

In [ ]:
cos = R @ R[0] / (np.linalg.norm(R, axis=1) * np.linalg.norm(R[0]))
giong_nhat = int(np.argmax(cos[1:]) + 1)
chua_xem = R[0] == 0
diem = np.where(chua_xem, R[giong_nhat], -1)          # phim đã xem thì cho -1 để không được chọn
goi_y = int(diem.argmax())
print(dict(zip(nguoi, cos.round(3))))
print(f"giống Lan nhất: {nguoi[giong_nhat]} → gợi ý {phim[goi_y]}")

{'Lan': 1.0, 'Minh': 0.811, 'Hoa': 0.272, 'Tuấn': 0.076}
giống Lan nhất: Minh → gợi ý Frozen


## Bài tập 5: Lớp Linear dự đoán giá nhà {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
X = np.array([[60, 2, 5.0],
              [85, 3, 8.0],
              [45, 1, 2.0],
              [120, 4, 12.0]])                 # 4 căn × 3 đặc trưng
W = np.array([[0.04, 0.3, -0.10],              # hàng 1: giá bán
              [0.12, 1.5, -0.40]])             # hàng 2: giá thuê
b = np.array([0.5, 2.0])
# TODO: Y = ...; in Y.shape

In [ ]:
Y = X @ W.T + b
print(X.shape, W.T.shape, Y.shape)
print(Y.round(2))

(4, 3) (3, 2) (4, 2)
[[ 3.  10.2]
 [ 4.  13.5]
 [ 2.4  8.1]
 [ 5.3 17.6]]


## Bài tập 6: Học điểm thi từ số giờ ôn bằng gradient descent {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
gio = np.array([1, 2, 3, 4, 5, 6, 7, 8], dtype=float)
diem = np.array([3.2, 4.1, 5.3, 5.8, 6.9, 7.4, 8.6, 9.0])

def loss(w, b):
    return np.mean((w * gio + b - diem) ** 2)

def grad(w, b):
    # TODO: trả về (dL/dw, dL/db)
    return 0.0, 0.0

In [ ]:
def grad(w, b):
    e = w * gio + b - diem
    return 2 * np.mean(e * gio), 2 * np.mean(e)

h = 1e-5
so = ((loss(h, 0) - loss(-h, 0)) / (2 * h), (loss(0, h) - loss(0, -h)) / (2 * h))
print("công thức:", grad(0, 0), " đạo hàm số:", so)

w, b, lr = 0.0, 0.0, 0.01
for step in range(2000):
    gw, gb = grad(w, b)
    w, b = w - lr * gw, b - lr * gb
print(f"gradient descent: w = {w:.3f}, b = {b:.3f}, loss = {loss(w, b):.4f}")
print("np.polyfit:      ", np.polyfit(gio, diem, 1).round(3))

công thức: (-65.39999999999999, -12.575)  đạo hàm số: (-65.39999999972679, -12.574999999870327)
gradient descent: w = 0.839, b = 2.510, loss = 0.0330
np.polyfit:       [0.839 2.511]


## Bài tập 7: Nhà vận chuyển nào đáng tin hơn {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import math
import numpy as np
A = np.array([46, 50, 48, 52, 47, 49, 51, 45, 53, 49], dtype=float)
B = np.array([38, 60, 41, 55, 36, 62, 40, 58, 39, 61], dtype=float)

def p_qua(nguong, mu, sigma):
    """P(X > nguong) với X ~ N(mu, sigma²)."""
    z = (nguong - mu) / sigma
    return 0.5 * (1 - math.erf(z / math.sqrt(2)))
# TODO: in mean, std của A, B; pA, pB = xác suất quá 60 giờ

In [ ]:
for ten, x in [("A", A), ("B", B)]:
    print(f"{ten}: trung bình {x.mean():.1f} giờ, độ lệch chuẩn {x.std():.2f}")
pA = p_qua(60, A.mean(), A.std())
pB = p_qua(60, B.mean(), B.std())
rng = np.random.default_rng(0)
mo_phong = (rng.normal(B.mean(), B.std(), 100_000) > 60).mean()
print(f"P(quá 60 giờ): A ≈ {pA:.4f}, B ≈ {pB:.3f}, mô phỏng B ≈ {mo_phong:.3f}")

A: trung bình 49.0 giờ, độ lệch chuẩn 2.45
B: trung bình 49.0 giờ, độ lệch chuẩn 10.42
P(quá 60 giờ): A ≈ 0.0000, B ≈ 0.146, mô phỏng B ≈ 0.144


## Bài tập 8: Lọc thư rác bằng Bayes {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
p_rac = 200 / 1000
p_km_rac, p_km_thuong = 120 / 200, 40 / 800
p_mp_rac, p_mp_thuong = 90 / 200, 16 / 800
# TODO: post_km, post_ca_hai

In [ ]:
p_thuong = 1 - p_rac
post_km = p_km_rac * p_rac / (p_km_rac * p_rac + p_km_thuong * p_thuong)
tu_rac = p_km_rac * p_mp_rac * p_rac
tu_thuong = p_km_thuong * p_mp_thuong * p_thuong
post_ca_hai = tu_rac / (tu_rac + tu_thuong)
print(f"P(rác | khuyến mãi) = {post_km:.3f}, P(rác | cả hai) = {post_ca_hai:.3f}")

P(rác | khuyến mãi) = 0.750, P(rác | cả hai) = 0.985


## Bài tập 9: Ước lượng tỉ lệ click bằng maximum likelihood {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
clicks = np.array([1] * 7 + [0] * 43)
ps = np.linspace(0.01, 0.99, 99)
# TODO: loglik (99 số), p_best, bce tại p_best

In [ ]:
k, n = clicks.sum(), len(clicks)
loglik = k * np.log(ps) + (n - k) * np.log(1 - ps)
p_best = ps[loglik.argmax()]
bce = -np.mean(clicks * np.log(p_best) + (1 - clicks) * np.log(1 - p_best))
print(f"p tốt nhất = {p_best:.2f}, công thức k/n = {k / n:.2f}")
print(f"BCE = {bce:.4f}, -loglik/n = {-loglik.max() / n:.4f}")

p tốt nhất = 0.14, công thức k/n = 0.14
BCE = 0.4050, -loglik/n = 0.4050
